# Step 1 — the Mindat point set

What `code/01_mindat_dataset.py` produced: every Mindat locality that has a
position and a mineral list, with duplicates at the same position merged.

This notebook only reads files. Re-run it any time; it never touches the
network and never rebuilds anything.

    python code/01_mindat_dataset.py --stage all


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "code"))
import common as C

# One fixed categorical order, used the same way in every chart and never
# cycled; a single hue carries magnitude. Validated for colour-vision
# deficiency, so series stay distinguishable.
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
SEQ = "Blues"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.figsize": (8, 4),
})

def report(name):
    """Load data/reports/<name>.json, or say which step writes it."""
    p = C.REPORTS / f"{name}.json"
    if not p.exists():
        raise SystemExit(f"{p.relative_to(REPO)} is missing — run the step that writes it.")
    return json.loads(p.read_text())

def need(path):
    p = Path(path)
    if not p.exists():
        raise SystemExit(f"{p} is missing — run the step that writes it first.")
    return p

def labelled_barh(ax, labels, values, color=CAT[0], fmt="{:,.0f}"):
    """Horizontal bars with the value written at the end of each one."""
    y = np.arange(len(labels))
    ax.barh(y, values, color=color, height=0.72)
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.xaxis.set_visible(False)
    ax.grid(False)
    span = max(values) if len(values) else 1
    for yi, v in zip(y, values):
        ax.text(v + span * 0.01, yi, fmt.format(v), va="center", fontsize=9, color=MUTED)
    for s in ("left", "bottom"):
        ax.spines[s].set_visible(False)
    return ax


In [ ]:
clean = report("01_clean")
labels = report("01_labels")
sites = pd.read_csv(need(C.SITES_CSV), low_memory=False)
print(f"{len(sites):,} sites, {clean['distinct_types']:,} locality types, "
      f"{labels['distinct_minerals']:,} distinct minerals")
sites.head(3)

## The funnel

Where the 451k raw localities went. Only two filters drop rows — no
coordinates (Mindat writes an unknown position as exactly `(0,0)`) and no
mineral list, because the mineral list *is* the label. Locality type filters
nothing: a point typed "mountain" can still carry a real mineral list.

In [ ]:
funnel = pd.DataFrame(clean["funnel"])
funnel["lost"] = funnel.rows.diff().fillna(0).astype(int)
display(funnel.style.format({"rows": "{:,}", "lost": "{:+,}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 3))
labelled_barh(ax, funnel.stage, funnel.rows)
ax.set_title("Localities surviving each filter", loc="left", pad=12)
plt.tight_layout()

## Locality types

Kept for information, not used as a filter. `is_mine` is a flag derived from
the type name, so it can be used to subset later without rebuilding.

In [ ]:
types = pd.DataFrame(clean["types"]).head(20)
fig, ax = plt.subplots(figsize=(8, 6))
labelled_barh(ax, [t or "(unnamed)" for t in types.type_name], types.sites)
ax.set_title(f"Sites by locality type (top 20 of {clean['distinct_types']:,})",
             loc="left", pad=12)
plt.tight_layout()

print(f"typed as mining-related: {clean['is_mine']:,} of {clean['sites']:,} "
      f"({100 * clean['is_mine'] / clean['sites']:.1f}%)")

## Where they are

One dot per site. The shape of the continents falling out of a mineral database is the quickest sanity check that the coordinates survived cleaning.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sites.lon, sites.lat, s=0.4, c=CAT[0], alpha=0.35, linewidths=0)
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="longitude", ylabel="latitude")
ax.set_aspect("equal")
ax.set_title(f"{len(sites):,} Mindat sites with coordinates and a mineral list",
             loc="left", pad=12)
plt.tight_layout()

## How many minerals per site

This is what makes the task multi-label: most sites list several minerals and
some list dozens. The label is the whole list.

In [ ]:
n = sites.n_minerals
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(n.clip(upper=40), bins=np.arange(0.5, 42), color=CAT[0])
ax.set(xlabel="minerals listed at the site (clipped at 40)", ylabel="sites")
ax.set_title(f"median {n.median():.0f}, mean {n.mean():.1f}, max {n.max():,}",
             loc="left", pad=12)
plt.tight_layout()

## The most common minerals

The head of this distribution is the usable label vocabulary; the tail is thousands of minerals seen a handful of times.

In [ ]:
counts = pd.read_csv(need(C.LABELS / "mineral_counts.csv"))
top = counts.head(30)
fig, ax = plt.subplots(figsize=(8, 8))
labelled_barh(ax, top.mineral_name, top.n_sites)
ax.set_title(f"Sites per mineral (top 30 of {len(counts):,})", loc="left", pad=12)
plt.tight_layout()

## Choosing the vocabulary threshold

Two different quantities, so two charts rather than two y-axes: how many
minerals clear the bar, and how many sites keep at least one label. The second
is the one that matters — a high threshold buys a clean vocabulary by throwing
sites away.

In [ ]:
sweep = pd.DataFrame(labels["sweep"])
display(sweep.style.format({"minerals": "{:,}", "sites_with_label": "{:,}",
                            "pct_sites": "{:.1f}%"}).hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, (col, lab) in zip(axes, [("minerals", "minerals in vocabulary"),
                                 ("pct_sites", "% of sites keeping a label")]):
    ax.plot(sweep.threshold, sweep[col], color=CAT[0], lw=2, marker="o", ms=5)
    ax.set(xscale="log", xlabel="minimum sites per mineral", ylabel=lab)
    chosen = labels["min_sites"]
    ax.axvline(chosen, color=CAT[1], lw=1.5, ls="--")
    ax.annotate(f"chosen: {chosen}", (chosen, ax.get_ylim()[1]),
                xytext=(4, -12), textcoords="offset points", fontsize=9, color=CAT[1])
plt.tight_layout()

## What got merged

A 64 px chip at 30 m spans about 1.9 km, so two localities a metre apart are
the same pixels. Keeping both would mean identical imagery under two different
labels — and the same ground in both the training and the test split. These
rows were merged into one site whose mineral list is the union.

In [ ]:
merges = pd.read_csv(need(C.REPORTS / "01_merges.csv"))
print(f"{len(merges):,} positions had more than one locality")
if len(merges):
    display(merges.sort_values("n_merged", ascending=False).head(15))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(merges.n_merged.clip(upper=12), bins=np.arange(1.5, 14),
            color=CAT[0])
    ax.set(xlabel="localities merged into one site (clipped at 12)", ylabel="positions")
    plt.tight_layout()